# Does perpetual futures flow and positioning improve 15-minute BTC spot prediction?

**Python for Data Science project report** · Antoine, Tina and Arya

This notebook is the project report. It describes every step of the project, from data collection to the model results, and runs all of them: each code cell calls functions from `src/`, the same ones `python -m src.pipeline` runs, so the text, the code and the results always agree.

**Running it.** Install the project as described in the README, then run every cell on a fresh kernel. All data comes from public Binance URLs, with no API key. On a fresh clone, the first cell of section 2 downloads about 31 MB, which takes about 5 minutes. After that, downloads and clean tables are reused from the gitignored `data/` folder, and the whole notebook runs in about 3 minutes, mostly for the models in section 8. Figures are saved to `report/figures` and tables to `report/tables`.

All numbers quoted in the text refer to BTCUSDT from 1 January to 30 June 2026, the window set in `config.yaml`.

**Contents**
1. Question, hypotheses and approach
2. Data collection
3. Cleaning and the 1-minute panel
4. Features
5. Descriptive analysis
6. Descriptive verdict
7. Targets, models and validation design
8. Results: block A vs A + B
9. Conclusion
10. Limitations
11. Reproducibility

## 1. Question, hypotheses and approach

**Question.** Does perpetual futures flow and positioning improve short-horizon BTC spot price prediction over a spot-only baseline?

**Motivation.** In crypto, a large share of trading and price discovery is thought to happen on perpetual futures ("perps") rather than on spot. Perps also carry information that spot cannot: leveraged positioning (open interest), the cost of holding a position (funding) and the gap between the perp and spot prices (basis). If that information reaches spot with a delay, it should improve spot forecasts.

**Hypotheses**, fixed before looking at any result:
- **H1:** adding perp features (block B) to spot-only features (block A) improves out-of-sample forecasts of the 15-minute BTC spot return.
- **H2:** perp-to-spot lead-lag is visible in the descriptive analysis (cross-correlations, Granger tests, event study) and fades within the 15-minute window.

**Scope.** One coin, BTC: BTCUSDT spot against the BTCUSDT USD-M perpetual on Binance, over six months, at one forecast horizon of 15 minutes. Instead of breadth, the work goes into a detailed descriptive analysis of how the two markets interact, and into a pipeline where the coin, horizon and window are settings in `config.yaml`, so the same code runs on any Binance pair and horizon without edits.

**Approach.** The core of the project is an ablation: every model is fitted once on the spot-only features and once with the perp features added, on the same minutes, and the difference answers H1. A null result is a valid finding as long as the design is sound, so much of the effort goes into avoiding look-ahead and testing the difference properly.

| Step | Section | Code | Output |
| --- | --- | --- | --- |
| Download from Binance | 2 | `src/fetch.py` | `data/raw/` |
| Clean, then align spot and perp minute by minute | 3 | `src/clean.py`, `src/panel.py` | `data/clean/`, `data/panel/` |
| Features (blocks A and B) | 4 | `src/features.py` | Columns added to the panel |
| Descriptive analysis | 5–6 | `src/descriptive.py`, `src/plots.py` | Figures 1–10, descriptive tables |
| Targets, models and tests | 7–8 | `src/models.py`, `src/evaluate.py`, `src/plots.py` | Figures 11–12, results tables |

In [ ]:
import pandas as pd
from IPython.display import Image, display

from src import clean, descriptive, fetch, plots
from src.config import load_config
from src.evaluate import run_ablation
from src.features import build_features, sensitivity_table
from src.models import make_targets
from src.panel import build_panel
from src.pipeline import feature_blocks

cfg = load_config()  # every run parameter: symbol, horizon, window, lags, seed, paths
symbol, horizon = cfg["symbol"], cfg["horizon_minutes"]
start, end = cfg["start_date"], cfg["end_date"]
print(f"{symbol}, {horizon}-minute horizon, {start} to {end}")

## 2. Data collection

All data comes from Binance public endpoints and is downloaded by the code itself. Klines (candles) record, for every minute, the volume bought by takers, the traders who cross the spread. So 1-minute klines already give the order flow and an average traded price, without downloading every trade:

$$\text{signed volume} = 2 \times \text{taker buy volume} - \text{volume} \qquad \mathrm{VWAP} = \frac{\text{quote volume}}{\text{volume}}$$

This keeps the download to about 31 MB, against gigabytes for tick data.

| Dataset | Market | Route | Resolution | Used for |
| --- | --- | --- | --- | --- |
| `klines` 1m | Spot | Archive | 1 min | Spot VWAP (the target), spot flow, volume, trade count |
| `klines` 1m | USD-M perp | Archive | 1 min | Perp VWAP, perp flow |
| `premiumIndexKlines` 1m | USD-M perp | Archive | 1 min | Kept for reference only: the basis is computed from the two VWAPs |
| `metrics` | USD-M perp | Archive | 5 min | Open interest, long/short and taker ratios |
| `fundingRate` | USD-M perp | REST API | 8 h | Funding rate |
| First and last candle, exchange info | Spot and perp | REST API | – | Availability check |

The data comes through two routes:
- **Archive** (`data.binance.vision`): zipped CSV files, one per month (one per day for `metrics`). Each download is checked against the SHA-256 checksum Binance publishes with it.
- **REST API**: the funding history (paginated) and the availability check. Responses are cached as raw JSON, so a re-run sends no new requests. Open interest is not taken from the REST API, which only serves the last 30 days.

Binance does not publish liquidations of USD-M perps in its archive (no `liquidationSnapshot` files, checked on 1 November 2026), so section 4 builds a proxy for them from open interest and price moves.

`fetch_all` first checks that both markets cover the whole window, and stops otherwise. It then downloads every file not already in `data/raw/` and logs it.

In [ ]:
files = fetch.fetch_all(cfg, symbol, start, end)
availability = pd.read_csv(cfg["data_dir"] / f"availability_{symbol}.csv")
availability[["market", "status", "first_candle_utc", "covers_window_start", "covers_window_end"]]

Both markets traded long before the window (spot since August 2017, the perp since September 2019) and are still trading, so the six months are fully available on both.

In [ ]:
display(files.groupby(["market", "dataset"]).size().rename("archive files").to_frame())
downloads = pd.read_csv(cfg["data_dir"] / "raw" / "download_log.csv")
downloads.groupby("status")["size_bytes"].agg(files="count", megabytes=lambda size: size.sum() / 1e6).round(1)

The archive gives 199 files: 6 monthly files each for spot klines, perp klines and the premium index, and 181 daily `metrics` files. All of them match their checksum (`verified`), for 30.7 MB in total. The 3 REST responses (exchange info for each market, and the 543 funding settlements) add 0.1 MB. `data/raw/download_log.csv` records the URL, size and SHA-256 of every file, so each number in this report can be traced back to its source.

## 3. Cleaning and the 1-minute panel

The data goes through three layers. Each is saved as parquet in `data/`, so later steps never redo earlier ones:

1. **Raw** (`data/raw/`): the files exactly as downloaded.
2. **Clean** (`data/clean/`, built by `src/clean.py`): one typed, deduplicated table per dataset, with every timestamp in UTC milliseconds.
3. **Panel** (`data/panel/`, built by `src/panel.py`): one row per minute, with spot and perp side by side.

Files are named by symbol and window, so runs with other settings never overwrite each other.

**Cleaning choices**
- **Headers.** Spot archive files have no header row and perp files do. Columns are always named by position, using the layouts in `src/schemas.py`, and a header that does not match the expected layout stops the run.
- **Timestamps.** Binance switched spot timestamps from milliseconds to microseconds on 1 January 2025. Each value is converted according to its own magnitude, so any window works. `metrics` date strings and funding times are converted to the same unit.
- **Window.** Monthly files can extend past the window; rows outside it are dropped.
- **Duplicates.** Repeated rows are dropped. If two different rows share a timestamp, the last one is kept and a warning is logged.
- **Types.** Prices, volumes and ratios are `float64`, trade counts `int64`. Unused columns are dropped, including the volume and count columns of `premiumIndexKlines`, which Binance fills with placeholders.
- **Check.** A candle that does not start on a whole minute stops the run, since it would mean a timestamp unit bug.

In [ ]:
clean.clean_all(cfg, symbol, start, end)

Every table covers the whole window: 260,640 one-minute candles on each market (181 days × 1,440 minutes), 52,128 `metrics` readings (181 × 288, one every 5 minutes) and 543 funding settlements (181 × 3, one every 8 hours). The only gap is in Binance's own premium index file for June, which has no rows for 29 June (259,200 = 180 × 1,440). No feature or figure uses the premium index, since the basis is computed from the two VWAPs, so this gap has no effect on the results.

**The panel.** Row *t* holds the candles of the minute starting at *t* on both markets, with their VWAP and signed volume, plus the latest `metrics` reading and funding rate. The slower datasets are the main risk of look-ahead, so they are joined **backward, as of the time each value becomes known**, never forward:

| Dataset | Timestamp | Known at |
| --- | --- | --- |
| `klines`, `premiumIndexKlines` | Start of the minute | End of the minute |
| `metrics` | `create_time`, start of its 5-minute window | `create_time` + 5 min |
| Funding | Settlement time | Settlement time |

Binance does not document what a `metrics` timestamp means, so we checked it. We rebuilt its taker buy/sell ratio from the perp klines of 1 January: it matches the 5 minutes *starting* at `create_time` (correlation 0.995), not the 5 minutes ending there (0.04). A reading stamped 12:00 therefore describes 12:00–12:05 and is only used from 12:05. Open interest, which could not be checked this way, gets the same conservative delay.

- **No silent filling.** Every minute of the window is a row. A missing candle stays empty and is flagged in a gap mask (`spot_missing`, `perp_missing`, `gap`). A `metrics` reading older than 15 minutes is left missing rather than carried through an outage.
- **Start.** The panel starts at 00:05 on the first day, the first minute at which a `metrics` reading is known.
- **Check.** The run stops if the median gap between the perp and spot closes is above 1%, which would mean a join or unit bug.

In [ ]:
panel = build_panel(cfg, symbol, start, end)
print(f"{len(panel):,} minutes; {int(panel['spot_missing'].sum())} missing on spot, "
      f"{int(panel['perp_missing'].sum())} missing on the perp")

# First minutes, with times in UTC: each metrics reading is only used once its 5-minute window has closed
times = ["open_time", "metrics_time", "funding_time"]
first_rows = panel[[*times, "spot_vwap", "perp_vwap", "sum_open_interest", "funding_rate"]].head(7).copy()
first_rows[times] = first_rows[times].apply(pd.to_datetime, unit="ms")
first_rows

The panel has 260,635 minutes (the window minus its first 5) and no missing minute on either market. The first rows show the backward join at work: the `metrics` reading stamped 00:00 is used from 00:05 to 00:09, and the one stamped 00:05 only from 00:10, once its window has closed. Section 5.1 looks at data quality in more detail.

## 4. Features

Features are split into two blocks. **Block A** uses spot data only and is the baseline. **Block B** adds the perp, and its added value is the answer to H1.

Rules for both blocks:
- **No look-ahead.** Candle features on row *t* only use minutes up to *t* − 1, and `metrics` and funding are only in the panel once known (section 3). A unit test checks that changing data after *t* never changes a feature at *t*.
- **Scale-free units.** Flows are fractions of volume, the basis is in basis points (bps, 0.01%), open interest is in % changes and ratios are in logs. Features stay comparable over time and across coins.
- **Horizon-aware lags.** The look-back windows *L* come from `config.yaml` (1, 5, 15 and 60 minutes for a 15-minute horizon) and are rescaled for another horizon. Block A uses all of them; block B uses 5 and 15, to stay near 15 features per block.
- **No filling.** A window containing a missing minute gives a missing feature.

**Block A: spot only (the baseline)**

| Name | Formula | Unit | Why |
| --- | --- | --- | --- |
| `a_ret_L` | log VWAP(*t*−1) − log VWAP(*t*−1−*L*) | log return | Spot momentum or reversal |
| `a_flow_L` | Σ signed volume / Σ volume over *L* | fraction (−1 to 1) | Spot buying pressure |
| `a_rv_L` | std of 1-min returns over *L* (*L* ≥ 5) | log return | Volatility regime |
| `a_count_rel`, `a_vol_rel` | trade count, volume / their 60-min mean | ratio | Unusual activity |
| `a_hour_sin`, `a_hour_cos` | sin, cos(2π · hour / 24) | – | Intraday cycle (section 5.1) |

**Block B: perp**

| Name | Formula | Unit | Why |
| --- | --- | --- | --- |
| `b_flow_L` | as `a_flow_L`, on the perp | fraction (−1 to 1) | Perp buying pressure may lead spot |
| `b_flow_div_L` | `b_flow_L` − `a_flow_L` | fraction | Perp pressure that spot has not shown yet |
| `b_basis_bps` | (perp VWAP / spot VWAP − 1) × 10⁴ | bps | Spot tends to close the gap |
| `b_basis_chg_L` | change in `b_basis_bps` over *L* | bps | Fresh perp pressure |
| `b_lead_L` | perp log return − spot log return over *L* | log return | The perp moved first; spot may follow |
| `b_oi_chg` | % change in open interest since the previous reading | % | Positions opened or closed |
| `b_toptrader_ls`, `b_ls`, `b_taker_ratio` | log of each `metrics` ratio | log ratio | Positioning skew |
| `b_funding_bps` | funding rate × 10⁴ | bps | Slow context on the crowded side |
| `b_delev_long_L`, `b_delev_short_L` | suspected deleveraging events by side that became known in the last *L* minutes | count | Forced perp selling or buying may spill into spot |

Section 5.3 shows that `b_flow_div_L` and `b_basis_chg_L` duplicate other features, so the models drop them.

**Deleveraging proxy.** Forced liquidations of leveraged perp positions are a natural channel from perp to spot, but Binance does not publish them (section 2). A 5-minute bar is flagged as **suspected deleveraging** when open interest falls by at least its 1% quantile **and** either the absolute perp return or the absolute perp flow share reaches its 95% quantile. The side is long when the price fell (longs forced to sell) and short when it rose. Events at most 30 minutes apart form one cluster. The descriptive analysis uses quantiles over the full sample; the model features only use quantiles of earlier bars, so they never see the future. A bar is only counted once its last open interest reading is known, 10 minutes after the bar starts.

In [ ]:
block_a, block_b = feature_blocks(cfg)
features = build_features(panel, cfg)

summary = features[block_a + block_b].agg(["count", "mean", "std", "min", "max"]).T
summary.insert(0, "block", ["A"] * len(block_a) + ["B"] * len(block_b))
summary.insert(1, "missing", len(features) - summary.pop("count").astype(int))
summary.round(4)

Block A has 15 features and block B 18. Missing values are few, and all at the start of the window: up to 61 minutes for the 60-minute windows, and about 1,470 minutes (one day) for the deleveraging counts, whose past-only thresholds need a day of history. The models drop these minutes (section 7). The values are plausible: flow shares stay between −1 and 1, the basis averages −4.9 bps, and the busiest minutes reach 28 times the hourly average trade count and 46 times the hourly average volume.

In [ ]:
sensitivity = sensitivity_table(panel)
descriptive.save_table(sensitivity, f"deleveraging_sensitivity_{symbol}", cfg)
sensitivity

Over the full sample, the 1% cut-off means a fall in open interest of at least 0.39% in 5 minutes, together with a perp move of at least 29.9 bps or a flow share of at least 0.46. It flags 161 suspected deleveraging events (98 with a falling price, 63 with a rising price), grouped into 127 clusters. The stricter 0.5% and looser 2% cut-offs give 91 and 279 events; section 5.4 checks that the event study holds at both. These are suspected events only: open interest also falls when traders close positions voluntarily.

## 5. Descriptive analysis

Before any model, this section asks whether the perp visibly leads spot and for how long (H2), and uses what it finds to justify the modelling choices of section 7. It covers data quality and market structure (5.1), returns and volatility (5.2), perp-specific signals (5.3) and lead-lag (5.4). Each figure is followed by what it shows and what it implies for the hypotheses or the models.

### 5.1 Data quality and market structure

In [ ]:
display(Image(plots.figure_1(descriptive.missing_minutes(panel), cfg)))

**Figure 1.** No minute is missing on either market over the window: the panel is complete, so no feature window is lost to gaps.

In [ ]:
tracking = descriptive.tracking_stats(panel)
descriptive.save_table(tracking, f"tracking_{symbol}", cfg)
tracking.round(2)

Perp and spot closes differ by a median 4.9 bps (99th percentile 7.0 bps, maximum 17 bps), so the join and the units are correct. The mean gap is −4.9 bps: the perp trades slightly below spot over the whole window.

In [ ]:
display(Image(plots.figure_2(descriptive.volume_share(panel), cfg)))

**Figure 2.** The perp carries 85–92% of the traded value every day (median 90%). Most BTC trading on Binance happens on the perp, which is why we look there for information that spot has not absorbed yet (H1, H2).

In [ ]:
display(Image(plots.figure_3(descriptive.intraday_profile(panel), cfg)))

**Figure 3.** Volume and volatility both peak at 14–15 UTC (US open and data releases) and are lowest around 04–10 UTC, on both markets. This intraday cycle is the reason for the time-of-day features in block A.

### 5.2 Returns and volatility

In [ ]:
samples = descriptive.return_samples(panel, cfg["horizon_minutes"])
moments = descriptive.return_moments(samples)
descriptive.save_table(moments, f"return_moments_{symbol}", cfg)
moments.round(2)

In [ ]:
display(Image(plots.figure_4(samples, cfg)))

**Figure 4.** Returns are strongly fat-tailed: excess kurtosis is 20.4 at 1 minute and 16.9 at 15 minutes (0 for a normal distribution), with 1-minute moves of up to 167 bps. The histograms are more peaked than the normal curve and the QQ plots bend away at both ends. A few large moves would dominate a squared-error fit, which supports the three-class target with a volatility-scaled threshold.

In [ ]:
display(Image(plots.figure_5(descriptive.acf_table(panel), cfg)))

**Figure 5.** Returns are uncorrelated from lag 2 on, apart from small negative values at lags 2–4. The lag-1 value of 0.21 is mechanical: averaging prices within each minute (VWAP) splits a move across two consecutive returns (Working, 1960). It also explains why the 15-minute std (25.9 bps) is above 5.8 × √15 ≈ 22.4 bps. Absolute returns stay strongly autocorrelated for an hour (0.38 at lag 1, still above 0.2 at lag 60): volatility clusters, which motivates the realised-volatility features. Part of this persistence comes from the intraday cycle of Figure 3.

In [ ]:
stationarity = descriptive.stationarity_tests(descriptive.five_minute_series(panel))
descriptive.save_table(stationarity, f"stationarity_{symbol}", cfg)
stationarity.round(4)

ADF rejects a unit root for all four series (5-minute bars). KPSS agrees for spot returns and perp flow, but flags the basis, whose mean drifts slowly, and, marginally, spot flow. All four can be used as features without differencing.

### 5.3 Perp-specific signals

In [ ]:
half_life = descriptive.basis_half_life(panel)
descriptive.save_table(half_life, f"basis_half_life_{symbol}", cfg)
clusters = descriptive.deleveraging_clusters(panel)
display(Image(plots.figure_6(descriptive.basis_over_time(panel), clusters, half_life, cfg)))
half_life.round(3)

**Figure 6.** The basis averages −4.9 bps. Minute-level deviations disappear within about a minute (AR(1) φ = 0.36, half-life 0.7 minutes), while the average level drifts slowly over weeks (KPSS above). The suspected deleveraging clusters are densest in early February.

In [ ]:
display(Image(plots.figure_7(descriptive.leverage_series(panel), cfg)))

**Figure 7.** Open interest falls from about 9 to 5.5 billion USDT in early February while funding turns negative until May. Part of the fall is the lower BTC price, since open interest is shown in USDT, but it coincides with the densest group of deleveraging clusters in Figure 6 (which measure open interest in BTC) and with the early-February 2026 sell-off, when BTC fell to about $60,000 on 6 February amid a broad unwind of leverage. Open interest rebuilds from March to mid-May and drops again in early June. The basis stays negative even when funding is positive: Binance keeps funding at its 0.01% default while its own premium index, measured against a multi-exchange spot index, stays between −0.04% and +0.06%.

In [ ]:
flows = descriptive.flow_correlation(panel, cfg["lags"])
descriptive.save_table(flows, f"flow_correlation_{symbol}", cfg)
flows.round(3)

Spot and perp signed flows correlate at about 0.5 (0.52 over 1 minute, 0.48 over 60 minutes): perp flow is far from a copy of spot flow, so it can carry information that block A lacks. Whether that extra part helps predict spot is what the models test.

In [ ]:
correlation = features[block_a + block_b].corr()
display(Image(plots.figure_10(correlation, cfg)))
redundant = descriptive.redundant_pairs(correlation)
descriptive.save_table(redundant, f"redundant_features_{symbol}", cfg)
redundant.round(3)

**Figure 10.** Within block A, the realised-volatility features are highly correlated (0.8–0.9). `b_basis_chg_L` duplicates `b_lead_L` (correlation 1.000), and `b_flow_div_L` is exactly `b_flow_L − a_flow_L`, a combination the pairwise check cannot flag. The models drop both kinds before fitting.

### 5.4 Lead-lag

In [ ]:
display(Image(plots.figure_8(descriptive.lead_lag_table(panel), cfg)))

**Figure 8.** The correlation of perp return, perp flow share and basis with the next minute's spot return is 0.25, 0.25 and 0.17, against 0.19, 0.05 and 0.07 with spot moving first; from two minutes on, every correlation is indistinguishable from zero. The grey band (±1.96/√N) assumes independent data and is too narrow, since returns are autocorrelated. Part of the one-minute correlation is mechanical (VWAP averaging, Figure 5); the asymmetry between +1 and −1 is the evidence of a lead.

In [ ]:
granger = descriptive.granger(panel)
descriptive.save_table(granger, f"granger_{symbol}", cfg)
granger.round(4)

With about 260,000 minutes every Granger test is significant, so the size of the effect matters: the perp's past adds 3.1–3.6% of explained variance to the next spot return (`delta_r2`), against at most 0.6% in the other direction. Granger causality is predictive, not causal.

In [ ]:
placebo = descriptive.placebo_events(panel, clusters)
study = descriptive.event_study(panel, clusters)
reference = descriptive.event_study(panel, placebo)
descriptive.save_table(study, f"event_study_{symbol}", cfg)
display(Image(plots.figure_9(study, reference, cfg)))

**Figure 9.** After the event bar, spot keeps moving in the direction of the flush for about 5 minutes and then stays flat: −27 bps after suspected long deleveraging and +18 bps after suspected short deleveraging, while equally large perp moves without an open interest drop slightly reverse. The basis barely moves (less than 1 bps). Caveats: a cluster can contain further deleveraging bars after time 0, windows of nearby clusters can overlap, and the placebo matches the size of the move but not the trend before it.

In [ ]:
# Appendix: the same event study at the 0.5% and 2% open interest thresholds
for oi_q, suffix in ((0.005, "_q0.5"), (0.02, "_q2")):
    appendix_clusters = descriptive.deleveraging_clusters(panel, oi_q)
    appendix_placebo = descriptive.placebo_events(panel, appendix_clusters)
    display(Image(plots.figure_9(descriptive.event_study(panel, appendix_clusters),
                                 descriptive.event_study(panel, appendix_placebo), cfg, suffix)))

**Appendix.** The direction is the same at both thresholds: stronger with the stricter 0.5% cut-off (−28 to −40 bps after long deleveraging; the 27 short clusters are too few for a clear band) and weaker at 2% (−15 and +10 bps), as expected if larger flushes matter more.

## 6. Descriptive verdict

The perp leads spot, but only by about one minute. The correlation of perp return, perp flow share and basis with the next minute's spot return is 0.25, 0.25 and 0.17, against 0.19, 0.05 and 0.07 in the other direction, and is indistinguishable from zero from two minutes on; the Granger tests agree (the perp's past adds 3.1–3.6% of explained variance to spot, at most 0.6% the other way), though part of the one-minute correlation is mechanical, from averaging prices within each minute. Around the 127 suspected deleveraging clusters, spot keeps moving in the direction of the flush for about five minutes after the event bar (−27 bps after long and +18 bps after short deleveraging), while equally large moves without an open interest drop do not continue, and this holds at the 0.5% and 2% thresholds. The lead therefore fades well within the 15-minute horizon. Since features only use minutes up to t − 1, the target starts at t, and deleveraging bars are only known five minutes after they close, we expect block B to add little or nothing over block A at 15 minutes: a weak or null result for H1 would be consistent with this evidence.

## 7. Targets, models and validation design

**Target.** The forward spot log return over the horizon *h* = 15 minutes, from 1-minute VWAPs:

$$y_t = \log \mathrm{VWAP}^{spot}_{t+h} - \log \mathrm{VWAP}^{spot}_{t}$$

Features use minutes up to *t* − 1 and the target starts at minute *t*, so the forecast never overlaps the information it is built from. A VWAP missing at *t* or *t* + *h* leaves the target missing; nothing is filled in.

**Why 15 minutes.** It is long enough to sit above trade-level noise (the spot archive has no historical bid and ask, and at second scale trade prices bounce between the two) and to use the 5-minute positioning data. It is short enough for a perp-leads-spot mechanism to be plausible, and it leaves about 260,000 forecasts, or 17,000 non-overlapping 15-minute windows. Daily returns would leave too few observations to detect small effects.

**Two tasks.**
- **Regression:** forecast *y* itself.
- **Classification:** down, flat or up. A return is flat when |*y*| ≤ *k* × `vol_scale`, where `vol_scale` is the standard deviation of 1-minute returns over the previous 60 minutes times √*h*, which is known when the forecast is made. *k* is fitted on each fold's training rows so that a third of them are flat. Returns are fat-tailed (Figure 4), so a few large moves dominate squared errors; the classes are not affected by them.

**Same minutes for both feature sets.** Minutes missing the target or any feature of A or B are dropped, so A and A + B are fitted and scored on exactly the same minutes and any difference comes from block B. The four duplicate block B features of section 5.3 are dropped first.

**Models.** Each is fitted once on A and once on A + B:

| Level | Regression | Classification | Why |
| --- | --- | --- | --- |
| 0 | Zero-return forecast (random walk) | Majority class (training shares) | The bar to beat: short-horizon returns are close to unpredictable |
| 1 | Ridge, lasso | Multinomial logistic regression | Interpretable; the coefficient table says which perp features matter |
| 2 | Histogram gradient boosting | Histogram gradient boosting | Non-linear effects and interactions the linear models miss |

Boosting has fixed settings suited to noisy minute data (200 trees, at least 500 rows per leaf, no early stopping), so a fit only depends on its training rows and the seed in `config.yaml`. Two model families were deliberately left out: polynomial regression, which has no motivation here and overfits, and deep learning (LSTM, transformers), which is hard to justify at this data scale and would turn the project into a benchmark race.

**Walk-forward validation.** No random splits: the window is cut into calendar months. Each fold tests on one month and trains on every month before it, from March to June, giving four test months.

| Fold | Fit (tuning) | Tune | Train (final fit) | Test |
| --- | --- | --- | --- | --- |
| 1 | January | February | January–February | March |
| 2 | January–February | March | January–March | April |
| 3 | January–March | April | January–April | May |
| 4 | January–April | May | January–May | June |

- **Tuning.** Hyperparameters (the ridge and lasso penalties, the logit `C`, the boosting depth and learning rate) are chosen by fitting on the *fit* months and scoring on the *tune* month. The best is then refitted on all training months and scored once on the test month, which is never used to choose anything.
- **Fitted on training rows only.** The feature scaler and the class threshold *k* are fitted on the rows a model trains on, then applied unchanged.
- **Embargo.** A label at *t* uses prices up to *t* + 15, so the last 15 minutes before the tune month and before the test month are dropped from the training rows.

**Metrics.** For regression, the out-of-sample R² against the zero forecast, 1 − SSE(model) / SSE(zero): 0 means no better than forecasting no move, and a negative value is worse. For classification, balanced accuracy, macro F1 and log loss, since plain accuracy rewards always predicting the most common class.

**Significance.** A forecast is made every minute for a 15-minute horizon, so consecutive forecast errors overlap and are autocorrelated. Every test therefore uses Newey-West (HAC) standard errors with 15 lags.
- **Clark-West** (one-sided) is the test of H1. Block A is nested in A + B, and when B's true coefficients are zero, estimating them adds noise to the larger model's forecasts; the test corrects for that noise.
- **Diebold-Mariano** (two-sided) compares non-nested models, such as ridge against boosting.
- **Holm correction** of the pooled Clark-West p-values across models, since several models are tested.

In [ ]:
# Regression target and volatility scale over the whole window, in bps
targets = make_targets(panel, horizon, cfg["target"]["vol_window_minutes"])
(targets[["y", "vol_scale"]] * 1e4).describe().round(2)

260,620 minutes have a target (the last 15 minutes of the window cannot). The 15-minute return has a mean of −0.2 bps and a standard deviation of 25.6 bps, so forecasting zero is a hard benchmark: any skill must come from small, predictable deviations around it. The volatility scale has a median of 16 bps but ranges from 1 to 155 bps, which is why the flat band moves with it rather than being fixed.

## 8. Results: block A vs A + B

The next cell runs the whole ablation of section 7: every model on A and on A + B, for both tasks and all four folds, then every test. It saves all the tables to `report/tables` and takes 2 to 3 minutes. The table it shows gives the size of each fold.

In [ ]:
results = run_ablation(cfg, features, block_a, block_b, symbol, start, end)
results["folds"][["fold", "test_month", "train_rows", "test_rows"]]

Each test month has between 43,185 and 44,640 forecasts, and the training window grows from 83,475 to 215,955 minutes. Training starts on 2 January at 00:30, after the day of history that the past-only deleveraging thresholds need (section 4).

In [ ]:
# Out-of-sample R² against the zero forecast, in %, across the four test months
regression = results["results_regression"].set_index(["model", "feature_set"])
(regression.drop(columns="folds") * 100).round(3)

Out-of-sample R² is measured against forecasting a zero return, so 0 means no skill. The best model is ridge on block A alone, with a mean of 0.034% (between −0.035% and 0.116% across months); adding block B lowers it to 0.001%. Lasso goes from 0.008% to −0.047%. Boosting is below zero with both feature sets (−0.13% and −0.16%): it fits noise in the training months, and the Diebold-Mariano test confirms that ridge beats it on block A (p = 0.013, table `diebold_mariano`). Fifteen-minute returns are close to unpredictable from these features, and block B makes every model slightly worse on average.

In [ ]:
classification = results["results_classification"].set_index(["model", "feature_set"])
classification[["balanced_accuracy_mean", "macro_f1_mean", "log_loss_mean"]].round(4)

The three classes (down, flat, up) are about equally frequent, by construction of the volatility-scaled threshold, so the majority class scores a balanced accuracy of 1/3. Logit and boosting reach 0.369–0.370 on block A and 0.364 on A + B, and their log loss (1.094–1.095) is only slightly below the 1.099 of the majority class. Block B again adds nothing.

In [ ]:
display(Image(plots.figure_11(results["clark_west"], cfg, folds=results["folds"])))
clark_west = results["clark_west"]
clark_west.assign(r2_gain=clark_west["r2_gain"] * 100).round(4)  # R² gain in percentage points, as in the figure

**Figure 11.** In March, ridge and boosting gain slightly from block B (+0.003 and +0.031 percentage points of R², one-sided Clark-West p = 0.07 and 0.19). Lasso drops every perp feature that month, so its two forecasts are identical. From April to June every model is worse with block B, and no fold reaches p < 0.05. Pooled over the four test months, the gains are −0.033 (ridge), −0.043 (lasso) and −0.029 (boosting) percentage points, with Holm-adjusted p-values of 1.00, 1.00 and 0.45. **H1 is not supported:** perp flow and positioning do not improve 15-minute spot forecasts out of sample. The Clark-West test corrects for the noise that the extra parameters add to the larger model, so this null result is not a penalty for adding features.

In [ ]:
display(Image(plots.figure_12(results["coefficients"], results["permutation_importance"], cfg)))
coefficients = results["coefficients"]
perp = coefficients[coefficients["block"] == "B"].sort_values("p_value")
perp.assign(**{c: perp[c] * 1e4 for c in ["coef", "hac_se", "ci_low", "ci_high"]}).round(3)  # in bps, as in the figure

**Figure 12.** Two perp coefficients have individual p-values near 0.03: the basis level (−0.48 bps of forecast 15-minute return per standard deviation) and the 5-minute perp lead (+0.27 bps). With 14 perp coefficients tested at once, about one such value is expected by chance, and after the Holm adjustment both are at 0.42. The two signs partly offset, since the 5-minute lead is close to the recent change in the basis. The deleveraging features have the widest intervals because the events are rare. Permutation importance tells the same story: no perp feature lowers test-month R² by more than 0.03 percentage points when shuffled, every range across folds includes zero, and many values are negative, meaning the forecast improves when the feature is scrambled. These coefficients are in-sample and descriptive, not causal.

## 9. Conclusion

**H2 is supported, but the lead is short.** The perp leads spot by about one minute. Perp returns, perp flow and the basis correlate more with the next minute's spot return than spot does with the perp's (Figure 8), and the perp's past adds 3.1–3.6% of explained variance to the next spot return, against at most 0.6% the other way (Granger). After suspected deleveraging, spot keeps moving in the direction of the flush for about five minutes (Figure 9). Every effect fades well within 15 minutes.

**H1 is not supported.** Adding the perp features does not improve 15-minute spot forecasts out of sample. No model gains on average over the four test months, no fold reaches 5% significance under the Clark-West test, and the pooled Holm-adjusted p-values are 0.45 or above (Figure 11). The classification models agree. The best regression model, ridge on block A, explains only 0.034% of the variance: 15-minute BTC returns are close to unpredictable from these features.

**Answer to the question.** At a 15-minute horizon, perp flow and positioning do not improve BTC spot prediction over a spot-only baseline. The two results are consistent with each other: the information that leads spot is absorbed within minutes, so by the time a 15-minute forecast is made, block A already contains what block B would add. This matches the expectation stated at the end of the descriptive verdict. The null result is informative rather than a failure of the design: the hypotheses were fixed in advance, both feature sets are scored on the same minutes with no look-ahead, and the test accounts for the noise the extra parameters add.

## 10. Limitations

- **One venue, one coin, one horizon.** The results describe Binance BTC at 15 minutes and may not hold for other coins, venues or horizons. Shorter horizons, where the one-minute lead could matter, are the natural next step; with this pipeline they are a change in `config.yaml`, but they were not part of this analysis.
- **Trade-based prices.** The spot archive has no historical bid and ask, so some microstructure noise remains at one minute. Averaging prices within each minute (VWAP) also creates a mechanical lag-1 correlation (Figure 5), which explains part of the one-minute lead.
- **Inferred liquidations.** The deleveraging proxy can flag voluntary position closing. Its thresholds were fixed in advance, and the event study holds at the 0.5% and 2% cut-offs, but the events remain suspected.
- **Predictive, not causal.** Granger tests, coefficients and permutation importances describe predictive relations, not causes.
- **No trading test.** A statistical gain, had there been one, need not survive trading fees and the bid-ask spread; no trading rule was evaluated.
- **One six-month regime.** The window includes a sharp sell-off in February and negative funding until May, and may not generalise to other periods. With four test months, the spread of results across folds rests on few points.

## 11. Reproducibility

- **One command.** After installing the project, `python -m src.pipeline` downloads the data and writes every table and figure of this report; this notebook calls the same functions. A fresh clone needs no manual step and no API key.
- **No data in Git.** `data/` and the generated figures and tables are gitignored and rebuilt from public Binance URLs, with checksums checked on every archive file.
- **Measured cost** (BTCUSDT, six months, one Windows laptop): a 31 MB download, about 7 minutes for a full pipeline run from an empty `data/` folder, and about 3 minutes for a re-run with the data already downloaded. `python -m src.pipeline --quick` runs on one month in about 10 seconds, without the models.
- **One configuration file.** The coin, horizon, window, lags, seed and paths are in `config.yaml`, nothing in `src/` hardcodes them, and every output is named by symbol and horizon. `--config` runs the pipeline on another file.
- **Tests.** `pytest` checks, among other things, that changing data after *t* never changes a feature at *t*, that `metrics` are used only once known, that spot timestamps are converted to one unit, that the embargo drops exactly *h* minutes, and the statistical tests.
- **Git workflow.** One branch per task, merged into `main` through pull requests reviewed by another team member.

## References

- Clark, T. E. and West, K. D. (2007). Approximately normal tests for equal predictive accuracy in nested models. *Journal of Econometrics*, 138(1), 291–311.
- Diebold, F. X. and Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
- Granger, C. W. J. (1969). Investigating causal relations by econometric models and cross-spectral methods. *Econometrica*, 37(3), 424–438.
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65–70.
- Newey, W. K. and West, K. D. (1987). A simple, positive semi-definite, heteroskedasticity and autocorrelation consistent covariance matrix. *Econometrica*, 55(3), 703–708.
- Working, H. (1960). Note on the correlation of first differences of averages in a random chain. *Econometrica*, 28(4), 916–918.